# 📊 AprovaEdu Analytics — Etapa 1 & 2: Extração e Tratamento dos Dados

**Desafio técnico AprovaEdu Analytics.** Base fictícia de uma rede de cursinhos pré-vestibular
(2021–2025), reunida de fontes internas e propositalmente "suja". Este notebook documenta,
com evidência nos dados, **cada decisão de tratamento** — requisito explícito do desafio.

O pipeline é reproduzível e dividido em camadas:

| Camada | Local | Conteúdo |
|---|---|---|
| Origem | `data/raw/` | XLSX do desafio (dicionário + amostras) |
| Bruto (CSV) | `data/processed/` | uma tabela por aba, **texto fiel** ao XLSX (`extract.py`) |
| Tratado | `data/final/` | base analítica tipada e normalizada (`transform.py`) |

**Perguntas obrigatórias** (respondidas na Etapa 3, próximo notebook):
1. Evolução da taxa de aprovação por ano.
2. Relação entre presença nas aulas e aprovação.
3. Cursos/matérias com melhor desempenho.
4. Recomendações para a coordenação.

## 1. Setup

In [1]:
import sys, re
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
pd.set_option("display.max_columns", None)

PROC = ROOT / "data" / "processed"
FINAL = ROOT / "data" / "final"
print("Raiz do projeto:", ROOT)

Raiz do projeto: C:\dev\data-analytics-portfolio


## 2. Extração — do XLSX para CSV (`extract.py`)

O XLSX traz o dicionário e **amostras** das tabelas. A extração lê cada aba e grava um CSV,
**sem alterar nada**: os valores são lidos como texto (`dtype=str`, `keep_default_na=False`),
preservando datas em formatos mistos, CPFs e inteiros com nulos. Assim a camada bruta é um
espelho fiel do arquivo e **todo o tratamento fica concentrado no `transform.py`** — nada é
"limpo" antes da hora. As abas de metadados (`Resumo`, `Problemas_Qualidade`) vão para
`data/processed/_meta/` por serem documentação, não dados.

In [2]:
from extract import extract_sheets, XLSX_PATH, OUTPUT_DIR

escritos = extract_sheets(XLSX_PATH, OUTPUT_DIR)
sorted(p.name for p in escritos.values() if "_meta" not in str(p))

📂 Lendo arquivo: base_pre_vestibular_dicionario_amostras.xlsx


📊 Abas encontradas: 11
----------------------------------------------------------------------
🗂  meta  Resumo                   → data\processed\_meta\resumo.csv            (11 linhas, 5 colunas)
🗂  meta  Problemas_Qualidade      → data\processed\_meta\problemas_qualidade.csv (6 linhas, 3 colunas)
✅ dado  Amostra_Professores      → data\processed\amostra_professores.csv     (35 linhas, 10 colunas)
✅ dado  Amostra_Estudantes       → data\processed\amostra_estudantes.csv      (500 linhas, 10 colunas)
✅ dado  Amostra_Ofertas_Curso    → data\processed\amostra_ofertas_curso.csv   (220 linhas, 13 colunas)
✅ dado  Amostra_Matriculas       → data\processed\amostra_matriculas.csv      (500 linhas, 10 colunas)
✅ dado  Amostra_Aprovacoes       → data\processed\amostra_aprovacoes.csv      (354 linhas, 11 colunas)
✅ dado  Amostra_Simulados        → data\processed\amostra_simulados.csv       (165 linhas, 11 colunas)
✅ dado  Amostra_Resultados_Sim   → data\processed\amostra_resultados_sim.csv  (500 l

['amostra_aprovacoes.csv',
 'amostra_aulas.csv',
 'amostra_estudantes.csv',
 'amostra_matriculas.csv',
 'amostra_ofertas_curso.csv',
 'amostra_presencas_aulas.csv',
 'amostra_professores.csv',
 'amostra_resultados_sim.csv',
 'amostra_simulados.csv']

### 2.1 Escopo dos dados — o XLSX é *dicionário + amostra*

Antes de perfilar, um alerta que condiciona toda a interpretação: o XLSX **não** é a base
completa. Verificamos isso **no próprio arquivo** (contando as linhas de cada aba com
`openpyxl`, independente do pandas/`extract.py`) e comparamos com o total declarado na aba
`Resumo`.

In [3]:
import openpyxl

XLSX = ROOT / "data" / "raw" / "base_pre_vestibular_dicionario_amostras.xlsx"

wb = openpyxl.load_workbook(XLSX)
linhas_no_xlsx = {ws.title: ws.max_row - 1 for ws in wb.worksheets}  # -1: cabeçalho
wb.close()

resumo = pd.read_excel(XLSX, sheet_name="Resumo", skiprows=2)
completo = dict(zip(resumo["Tabela"].str.strip(),
                    resumo["Linhas no CSV completo"].astype(int)))

escopo = pd.DataFrame([
    {"tabela": t, "linhas_no_xlsx": n, "base_completa": completo[t],
     "situacao": "Completa" if n >= completo[t] else f"Truncada em {n}"}
    for aba, n in linhas_no_xlsx.items()
    if aba.startswith("Amostra_") and (t := aba.replace("Amostra_", ""))
])
escopo

,tabela,linhas_no_xlsx,base_completa,situacao
0,Professores,35,35,Completa
1,Estudantes,500,812,Truncada em 500
2,Ofertas_Curso,220,220,Completa
3,Matriculas,500,9452,Truncada em 500
4,Aprovacoes,354,354,Completa
5,Simulados,165,165,Completa
6,Resultados_Sim,500,21510,Truncada em 500
7,Aulas,500,2418,Truncada em 500
8,Presencas_Aulas,500,74997,Truncada em 500


**Conclusões:**
1. **A limitação está no arquivo, não no `extract.py`** — a extração lê todas as linhas
   presentes (sem `nrows`/`skiprows`); extrai 500 porque 500 é o que o arquivo contém.
2. **Truncamento por ordem, não amostra aleatória** — nas tabelas cortadas, os IDs são os
   primeiros sequenciais (`M0000001…M0000500`), então cobrem fatias de entidades quase
   disjuntas.
3. **É por design** — a aba `Resumo` declara: *"A planilha traz amostra; o CSV contém a
   base completa"*. A base completa é distribuída à parte, como CSVs.

Consequência: os cruzamentos entre matrícula, presença e aprovação (base das perguntas)
ficam com sobreposição mínima — por isso o `02_analise.ipynb` é uma **demonstração de
método**. Ver `README.md` › *Escopo dos dados*.

## 3. Perfilamento — a evidência dos problemas

Antes de decidir *como* tratar, é preciso *ver* a sujeira. Carregamos os CSVs como texto puro
e medimos cada categoria de problema listada na aba `Problemas_Qualidade`.

In [4]:
def ler_bruto(nome):
    return pd.read_csv(PROC / f"amostra_{nome}.csv", dtype=str, keep_default_na=False)

tabelas_raw = ["professores","estudantes","ofertas_curso","matriculas","aprovacoes",
               "simulados","resultados_sim","aulas","presencas_aulas"]
{t: ler_bruto(t).shape for t in tabelas_raw}

{'professores': (35, 10),
 'estudantes': (500, 10),
 'ofertas_curso': (220, 13),
 'matriculas': (500, 10),
 'aprovacoes': (354, 11),
 'simulados': (165, 11),
 'resultados_sim': (500, 12),
 'aulas': (500, 10),
 'presencas_aulas': (500, 6)}

### 3.1 Datas em formatos mistos
Na mesma coluna convivem ISO, `aaaa/mm/dd`, `dd/mm/aaaa`, `dd-mm-aaaa` (e datetimes).

In [5]:
def formato_data(v):
    v = v.strip()
    if v == "": return "vazio"
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", v): return "ISO aaaa-mm-dd"
    if re.fullmatch(r"\d{4}/\d{2}/\d{2}", v): return "aaaa/mm/dd"
    if re.fullmatch(r"\d{2}/\d{2}/\d{4}", v): return "dd/mm/aaaa (com /)"
    if re.fullmatch(r"\d{2}-\d{2}-\d{4}", v): return "dd-mm-aaaa (com -)"
    if re.search(r"\d{2}:\d{2}", v): return "com hora"
    return "outro"

ler_bruto("estudantes")["data_nascimento"].map(formato_data).value_counts()

data_nascimento
ISO aaaa-mm-dd        459
dd-mm-aaaa (com -)     19
dd/mm/aaaa (com /)     13
aaaa/mm/dd              9
Name: count, dtype: int64

### 3.2 Categorias inconsistentes
Caixa, acento e abreviação divergentes para a mesma matéria.

In [6]:
ler_bruto("ofertas_curso")["materia"].value_counts()

materia
Português     20
Sociologia    20
Física        20
Geografia     20
Matemática    19
Inglês        19
Filosofia     19
História      19
Biologia      19
Redação       18
Química       18
Redacao        2
QUÍMICA        1
MATEMÁTICA     1
BIOLOGIA       1
Historia       1
Quimica        1
Ingles         1
FILOSOFIA      1
Name: count, dtype: int64

### 3.3 Valores faltantes (por coluna)

In [7]:
mat = ler_bruto("matriculas")
(mat == "").sum().sort_values(ascending=False).head(8)

origem_captacao      84
bolsa_percentual     45
status_matricula     17
matricula_id          0
aluno_id              0
oferta_id             0
ano                   0
materia_declarada     0
dtype: int64

### 3.4 Outliers — nota de simulado (escala 0–100)

In [8]:
nota = pd.to_numeric(ler_bruto("resultados_sim")["nota"], errors="coerce")
print("máximo observado:", nota.max(), "| notas > 100:", int((nota > 100).sum()))
nota.describe()

máximo observado: 105.0 | notas > 100: 5


count    461.000000
mean      60.965076
std       17.097805
min       16.700000
25%       49.000000
50%       59.900000
75%       72.500000
max      105.000000
Name: nota, dtype: float64

### 3.5 Duplicidade — sinal explícito

Em `aprovacoes`, as linhas duplicadas por **(aluno, universidade, curso, ano)** coincidem
**exatamente** com as marcadas `chamada = "Cadastro duplicado?"`. Isso dá uma regra de
deduplicação objetiva. Já alunos aprovados 2+ vezes em cursos/universidades **diferentes**
são legítimos e **não** devem ser removidos.

In [9]:
ap = ler_bruto("aprovacoes")
chave = ["aluno_id", "universidade", "curso_aprovado", "ano_vestibular"]
print("duplicadas por chave de negócio:", int(ap.duplicated(chave).sum()))
print('marcadas "Cadastro duplicado?":  ', int((ap["chamada"] == "Cadastro duplicado?").sum()))
print("alunos aprovados 2+ vezes (legítimo):", int((ap["aluno_id"].value_counts() >= 2).sum()))

duplicadas por chave de negócio: 15
marcadas "Cadastro duplicado?":   15
alunos aprovados 2+ vezes (legítimo): 46


## 4. Decisões de tratamento

| Problema | Decisão | Justificativa |
|---|---|---|
| **Datas mistas** | Converter para `datetime`; ambiguidade `dd/mm` vs `mm/dd` resolvida pela magnitude dos campos; caso ambíguo → **dd/mm** | Contexto brasileiro; `03-16` só pode ser `mm-dd`, `15/11` só `dd/mm` |
| **Categorias** | Dicionário canônico por *fold* (minúsculo, sem acento, sem pontuação) | Unifica `MATEMÁTICA`/`Matematica`/`Mat.` sem perder a forma final acentuada |
| **Denormalização** | `professor_id` é a FK; nome informado é conferido com a dimensão e **removido** | Dimensão `Professores` é a fonte de verdade |
| **Duplicidade** | Remover linhas `chamada = "Cadastro duplicado?"` em Aprovações | Coincidem 1:1 com duplicatas por chave de negócio |
| **Outliers** | `nota` fora de `[0,100]` → nula; tempo/acertos inconsistentes → apenas sinalizados | Não inventar valor; preservar o registro para auditoria |
| **Faltantes** | Medidas (notas) **não** imputadas; categóricas → `"Não informado"` | Imputar nota enviesaria as análises |
| **CPF** | Reduzir a 11 dígitos (forma canônica) | Padroniza `625.588.977-17` e `89233633131` |

As regras vivem em `src/cleaning.py` (funções puras + dicionários) e são orquestradas por
`src/transform.py`.

## 5. Execução do tratamento (`transform.py`)

In [10]:
from transform import main

tabelas = main()

# Relatório de tratamento — AprovaEdu Analytics

Gerado por `src/transform.py`. Correções aplicadas sobre `data/processed/` e base tratada salva em `data/final/`.

- **Ofertas – denormalização:** 0 nomes de professor divergentes do cadastro; usada a dimensão como fonte de verdade e removida a coluna informada.
- **Aprovações – duplicidade:** removidas 15 linhas marcadas "Cadastro duplicado?" (354 → 339).
- **Simulados – denormalização:** 0 nomes de professor divergentes do cadastro; usada a dimensão como fonte de verdade e removida a coluna informada.
- **Resultados – outliers de nota:** 5 notas fora de [0,100] convertidas para nula.
- **Resultados – consistência:** 0 com acertos > total de questões; 35 com tempo acima do limite (mantidos e sinalizados).

## Validação da base tratada (Pandera)



- ✅ Todas as tabelas passaram: PKs únicas e não-nulas, faixas numéricas plausíveis e categorias dentro do conjunto canônico.

## Tabelas geradas

- `professores`: 35 linhas × 10 colunas
- `estudantes`: 500 linhas × 10 colunas
- `ofertas_curso`: 220 linhas × 12 colunas
- `matriculas`: 500 linhas × 10 colunas
- `aprovacoes`: 339 linhas × 11 colunas
- `simulados`: 165 linhas × 10 colunas
- `resultados_sim`: 500 linhas × 12 colunas
- `aulas`: 500 linhas × 10 colunas
- `presencas_aulas`: 500 linhas × 6 colunas

🎉 Tratamento concluído! Base tratada em data\final


O `transform.py` grava um relatório com as contagens de cada correção:

In [11]:
print((FINAL / "_relatorio_tratamento.md").read_text(encoding="utf-8"))

# Relatório de tratamento — AprovaEdu Analytics

Gerado por `src/transform.py`. Correções aplicadas sobre `data/processed/` e base tratada salva em `data/final/`.

- **Ofertas – denormalização:** 0 nomes de professor divergentes do cadastro; usada a dimensão como fonte de verdade e removida a coluna informada.
- **Aprovações – duplicidade:** removidas 15 linhas marcadas "Cadastro duplicado?" (354 → 339).
- **Simulados – denormalização:** 0 nomes de professor divergentes do cadastro; usada a dimensão como fonte de verdade e removida a coluna informada.
- **Resultados – outliers de nota:** 5 notas fora de [0,100] convertidas para nula.
- **Resultados – consistência:** 0 com acertos > total de questões; 35 com tempo acima do limite (mantidos e sinalizados).

## Validação da base tratada (Pandera)

- ✅ Todas as tabelas passaram: PKs únicas e não-nulas, faixas numéricas plausíveis e categorias dentro do conjunto canônico.

## Tabelas geradas

- `professores`: 35 linhas × 10 colunas
- `estud

## 6. Validação da base tratada

### 6.1 Categorias: antes → depois (matéria em ofertas)

In [12]:
antes = sorted(ler_bruto("ofertas_curso")["materia"].unique())
depois = sorted(tabelas["ofertas_curso"]["materia"].dropna().unique())
print(f"{len(antes)} variações sujas → {len(depois)} categorias canônicas")
print("depois:", depois)

19 variações sujas → 11 categorias canônicas
depois: ['Biologia', 'Filosofia', 'Física', 'Geografia', 'História', 'Inglês', 'Matemática', 'Português', 'Química', 'Redação', 'Sociologia']


### 6.2 Tipos corretos e datas normalizadas

In [13]:
tabelas["aprovacoes"].dtypes

aprovacao_id                     object
ano_vestibular                    Int64
aluno_id                         object
universidade                     object
curso_aprovado                   object
modalidade_vaga                  object
chamada                          object
bolsa_aprovacao                  object
data_resultado           datetime64[ns]
nota_final_vestibular           float64
campus                           object
dtype: object

### 6.3 Outliers de nota removidos (máx. ≤ 100)

In [14]:
tabelas["resultados_sim"]["nota"].describe()

count    456.000000
mean      60.496930
std       16.591162
min       16.700000
25%       48.950000
50%       59.550000
75%       72.200000
max       99.400000
Name: nota, dtype: float64

## 7. Próximos passos

Com a base tratada em `data/final/` (Parquet + CSV), a **Etapa 3** responderá às 4 perguntas
obrigatórias — taxa de aprovação por ano, presença × aprovação, desempenho por matéria/curso —
e consolidará as recomendações para a coordenação, com apoio de um dashboard.